# Topic 2: RDD Transformations

> **Phase 2 → Week 2 → Topic 2**

---

## The Assembly Line Analogy

Imagine a car factory assembly line:
- Raw steel sheets enter → `sc.parallelize(raw_materials)`
- Station 1: cut into shapes → `.map(cut)` — narrow, one part becomes one part
- Station 2: weld frames → `.map(weld)` — still narrow
- Quality check: reject defective parts → `.filter(is_good)` — narrow
- Station 3: split one part into sub-parts → `.flatMap(split)` — narrow

Every station transforms the product into something new without changing the previous stage.
The production line only RUNS when the final product is ordered (Action called).

---

## All RDD Transformations You Need to Know

### Narrow Transformations (no shuffle)

| Transformation | Input → Output | Description |
|---------------|---------------|-------------|
| `map(f)` | 1 element → 1 element | Apply function to every element |
| `flatMap(f)` | 1 element → N elements | Apply function, flatten result |
| `filter(f)` | 1 element → 0 or 1 | Keep elements where f returns True |
| `mapValues(f)` | (k,v) → (k, f(v)) | Apply function to values only (pair RDD) |
| `keys()` | (k,v) → k | Extract keys from pair RDD |
| `values()` | (k,v) → v | Extract values from pair RDD |
| `union(other)` | two RDDs → one | Combine two RDDs |
| `sample(f, rate)` | RDD → subset | Random sample |
| `mapPartitions(f)` | partition → partition | Apply function to each partition as iterator |
| `mapPartitionsWithIndex(f)` | (idx, partition) → partition | Same + partition index |
| `coalesce(n)` | N parts → n parts | Reduce partitions without shuffle |

### Wide Transformations (shuffle)

| Transformation | Description |
|---------------|-------------|
| `distinct()` | Remove duplicates (full shuffle) |
| `repartition(n)` | Increase/decrease partitions with shuffle |
| `sortBy(f)` | Global sort (shuffle) |
| `intersection(other)` | Common elements (shuffle) |
| `subtract(other)` | Elements in this but not other (shuffle) |
| `cartesian(other)` | All combinations — expensive! |

---

## `map()` vs `flatMap()` — The Most Important Distinction

This is the #1 confused concept in RDD interviews.

```
map():     ["hello world", "spark rocks"]
           → ["hello world".split(), "spark rocks".split()]
           → [["hello", "world"], ["spark", "rocks"]]   ← nested lists!

flatMap(): ["hello world", "spark rocks"]
           → flatten(["hello", "world"], ["spark", "rocks"])
           → ["hello", "world", "spark", "rocks"]        ← flat list!
```

**Rule**: Use `flatMap` when your function returns a list and you want the
result to be a flat list (not a list of lists).

**Most common use**: `flatMap(lambda line: line.split())` for word tokenization.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week2 - RDD Transformations") \
    .master("local[4]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("SparkSession ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 23:30:16 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 10.237.41.61 instead (on interface wlan0)
26/06/09 23:30:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/09 23:30:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready!


## Part 1: `map()` — One In, One Out

In [2]:
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 2)

# map: apply function to every element
squared     = numbers.map(lambda x: x ** 2)
string_nums = numbers.map(lambda x: str(x))
tuples      = numbers.map(lambda x: (x, x ** 2))

print("Original:  ", numbers.collect())
print("Squared:   ", squared.collect())
print("As string: ", string_nums.collect())
print("As tuples: ", tuples.collect())

Original:   [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Squared:    [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
As string:  ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']


As tuples:  [(1, 1), (2, 4), (3, 9), (4, 16), (5, 25), (6, 36), (7, 49), (8, 64), (9, 81), (10, 100)]


In [3]:
# map with a regular function (not lambda) — better for complex logic
def categorize(n):
    if n <= 3:
        return (n, "low")
    elif n <= 7:
        return (n, "mid")
    else:
        return (n, "high")

categorized = numbers.map(categorize)
print("Categorized:", categorized.collect())

Categorized: [(1, 'low'), (2, 'low'), (3, 'low'), (4, 'mid'), (5, 'mid'), (6, 'mid'), (7, 'mid'), (8, 'high'), (9, 'high'), (10, 'high')]


In [4]:
# Real-world: parse log lines with map()
logs = sc.parallelize([
    "2024-01-01 ERROR Connection timeout on port 5432",
    "2024-01-02 INFO  User alice logged in",
    "2024-01-02 ERROR Disk full on /dev/sda1",
    "2024-01-03 WARN  High memory: 92% used",
    "2024-01-03 INFO  Backup completed successfully",
], 2)

def parse_log(line):
    parts = line.split(" ", 2)  # split only first 2 spaces
    return {"date": parts[0], "level": parts[1].strip(), "message": parts[2]}

parsed = logs.map(parse_log)
print("Parsed logs:")
for entry in parsed.collect():
    print(f"  {entry}")

Parsed logs:


  {'date': '2024-01-01', 'level': 'ERROR', 'message': 'Connection timeout on port 5432'}
  {'date': '2024-01-02', 'level': 'INFO', 'message': ' User alice logged in'}
  {'date': '2024-01-02', 'level': 'ERROR', 'message': 'Disk full on /dev/sda1'}
  {'date': '2024-01-03', 'level': 'WARN', 'message': ' High memory: 92% used'}
  {'date': '2024-01-03', 'level': 'INFO', 'message': ' Backup completed successfully'}


## Part 2: `flatMap()` — One In, Many Out (Flattened)

In [5]:
sentences = sc.parallelize([
    "apache spark is fast",
    "spark uses lazy evaluation",
    "rdd is the foundation",
], 2)

# map returns a list per element — gives nested list!
map_result = sentences.map(lambda s: s.split())
print("map().collect():", map_result.collect())
print("  → nested lists! Each sentence becomes a list")

# flatMap flattens the result
flatmap_result = sentences.flatMap(lambda s: s.split())
print("\nflatMap().collect():", flatmap_result.collect())
print("  → flat list! All words in one list")

map().collect(): [['apache', 'spark', 'is', 'fast'], ['spark', 'uses', 'lazy', 'evaluation'], ['rdd', 'is', 'the', 'foundation']]
  → nested lists! Each sentence becomes a list



flatMap().collect(): ['apache', 'spark', 'is', 'fast', 'spark', 'uses', 'lazy', 'evaluation', 'rdd', 'is', 'the', 'foundation']
  → flat list! All words in one list


In [6]:
# Classic word count with flatMap
word_pairs = sentences.flatMap(lambda s: s.split()) \
                       .map(lambda w: (w, 1)) \
                       .reduceByKey(lambda a, b: a + b) \
                       .sortBy(lambda x: x[1], ascending=False)

print("Word counts:")
for word, count in word_pairs.collect():
    print(f"  {word:<15} {count}")

Word counts:


  spark           2
  is              2
  apache          1
  fast            1
  uses            1
  lazy            1
  rdd             1
  evaluation      1
  the             1
  foundation      1


In [7]:
# flatMap for expanding ranges — generate multiple outputs per input
numbers_2 = sc.parallelize([1, 2, 3], 1)

# generate range for each number: 1→[1], 2→[1,2], 3→[1,2,3]
expanded = numbers_2.flatMap(lambda x: range(1, x + 1))
print("flatMap to expand ranges:")
print(f"  Input:  {numbers_2.collect()}")
print(f"  Output: {expanded.collect()}  (1→[1], 2→[1,2], 3→[1,2,3] flattened)")

flatMap to expand ranges:
  Input:  [1, 2, 3]
  Output: [1, 1, 2, 1, 2, 3]  (1→[1], 2→[1,2], 3→[1,2,3] flattened)


## Part 3: `filter()` — Keep or Discard

In [8]:
numbers = sc.parallelize(range(1, 21), 4)

# Basic filters
evens     = numbers.filter(lambda x: x % 2 == 0)
odd_gt_10 = numbers.filter(lambda x: x % 2 != 0 and x > 10)

print("Original (1-20):  ", numbers.collect())
print("Evens:            ", evens.collect())
print("Odd AND > 10:     ", odd_gt_10.collect())

Original (1-20):   [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Evens:             [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


Odd AND > 10:      [11, 13, 15, 17, 19]


In [9]:
# Real-world: filter log lines by level
logs_rdd = sc.parallelize([
    "2024-01-01 ERROR Connection timeout",
    "2024-01-01 INFO  User logged in",
    "2024-01-02 ERROR Disk full",
    "2024-01-02 WARN  High memory",
    "2024-01-03 INFO  Backup done",
    "2024-01-03 ERROR DB down",
], 3)

errors = logs_rdd.filter(lambda line: "ERROR" in line)
print(f"Total lines: {logs_rdd.count()}")
print(f"Error lines: {errors.count()}")
print("Error entries:")
for line in errors.collect():
    print(f"  {line}")

Total lines: 6


Error lines: 3
Error entries:


  2024-01-01 ERROR Connection timeout
  2024-01-02 ERROR Disk full
  2024-01-03 ERROR DB down


## Part 4: `distinct()` — Remove Duplicates

`distinct()` is a **wide transformation** — it requires a shuffle to ensure
all occurrences of the same value end up on the same executor.

In [10]:
dupes = sc.parallelize([1, 2, 2, 3, 3, 3, 4, 4, 5, 1, 2], 3)

unique = dupes.distinct()
print(f"Original ({dupes.count()} elements): {dupes.collect()}")
print(f"Distinct ({unique.count()} elements): {sorted(unique.collect())}")
print()
# distinct() triggers a shuffle — you can see it in the lineage
print("Lineage (notice ShuffledRDD):")
print(dupes.distinct().toDebugString().decode())

Original (11 elements): [1, 2, 2, 3, 3, 3, 4, 4, 5, 1, 2]


Distinct (5 elements): [1, 2, 3, 4, 5]

Lineage (notice ShuffledRDD):
(3) PythonRDD[43] at RDD at PythonRDD.scala:58 []
 |  MapPartitionsRDD[42] at mapPartitions at PythonRDD.scala:170 []
 |  ShuffledRDD[41] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[40] at distinct at /tmp/ipykernel_45675/1979005192.py:9 []
    |  PythonRDD[39] at distinct at /tmp/ipykernel_45675/1979005192.py:9 []
    |  ParallelCollectionRDD[31] at readRDDFromFile at PythonRDD.scala:299 []


In [11]:
# Real-world: find unique countries from user records
users = sc.parallelize([
    ("alice",  "India"),
    ("bob",    "USA"),
    ("carol",  "India"),
    ("dave",   "UK"),
    ("eve",    "India"),
    ("frank",  "USA"),
    ("grace",  "Australia"),
])

countries = users.map(lambda x: x[1]).distinct()
print("Unique countries:", sorted(countries.collect()))

Unique countries: ['Australia', 'India', 'UK', 'USA']


## Part 5: `union()` — Combine Two RDDs

`union()` is a **narrow transformation** — no shuffle needed.
It just combines the partitions of both RDDs.

**Gotcha**: `union()` does NOT remove duplicates. Use `.union().distinct()` if needed.

In [12]:
jan_sales = sc.parallelize([100, 200, 300, 150], 2)
feb_sales = sc.parallelize([250, 180, 320, 200], 2)  # 200 appears in both!

# Union: combine all sales
all_sales = jan_sales.union(feb_sales)
print(f"Jan sales ({jan_sales.count()} entries): {jan_sales.collect()}")
print(f"Feb sales ({feb_sales.count()} entries): {feb_sales.collect()}")
print(f"All sales ({all_sales.count()} entries): {all_sales.collect()}")
print(f"Partitions after union: {all_sales.getNumPartitions()}")
print()
print("Notice: 200 appears TWICE (union does NOT dedup)")
print("For dedup: all_sales.distinct().collect() =", sorted(all_sales.distinct().collect()))

Jan sales (4 entries): [100, 200, 300, 150]


Feb sales (4 entries): [250, 180, 320, 200]


All sales (8 entries): [100, 200, 300, 150, 250, 180, 320, 200]
Partitions after union: 4

Notice: 200 appears TWICE (union does NOT dedup)


For dedup: all_sales.distinct().collect() = [100, 150, 180, 200, 250, 300, 320]


In [13]:
# Real-world: union multiple log files from different days
day1_logs = sc.parallelize(["ERROR: timeout", "INFO: start", "ERROR: disk full"], 1)
day2_logs = sc.parallelize(["INFO: backup", "WARN: high mem", "ERROR: db down"], 1)
day3_logs = sc.parallelize(["INFO: done", "ERROR: cpu spike"], 1)

# Combine all days and filter errors
all_logs = day1_logs.union(day2_logs).union(day3_logs)
all_errors = all_logs.filter(lambda l: l.startswith("ERROR"))

print(f"Total log lines across 3 days: {all_logs.count()}")
print(f"Total errors: {all_errors.count()}")
print("Errors:")
for e in all_errors.collect():
    print(f"  {e}")

Total log lines across 3 days: 8


Total errors: 4
Errors:


  ERROR: timeout
  ERROR: disk full
  ERROR: db down
  ERROR: cpu spike


## Part 6: `mapPartitions()` — Process a Whole Partition at Once

This is a performance optimization technique. Instead of calling your function
for EVERY element, `mapPartitions` calls it ONCE per partition with an iterator.

**When to use:**
- Opening a database connection per partition (not per row)
- Heavy initialization that you want to do once per partition
- ML model inference (load model once per partition)

```
map():            open_conn() → write(row1) → close_conn()
                  open_conn() → write(row2) → close_conn()  ← 1M conn opens!
                  ...

mapPartitions(): open_conn() → write(row1, row2, ... rowN) → close_conn()
                 ← 1 conn open per partition (e.g., 8 partitions = 8 conn opens)
```

In [14]:
data = sc.parallelize(range(1, 13), 4)  # 12 elements, 4 partitions

def process_partition(iterator):
    """Simulates opening a DB connection once per partition."""
    items = list(iterator)
    # In production: conn = get_db_connection()  # once per partition
    # for item in items:
    #     conn.execute(insert_sql, item)
    # conn.close()
    
    # For demo: just compute the partition sum and label it
    return iter([(f"partition_result", sum(items), len(items))])

results = data.mapPartitions(process_partition).collect()
print("mapPartitions results (one tuple per partition):")
for r in results:
    print(f"  key={r[0]}, sum={r[1]}, count={r[2]}")

mapPartitions results (one tuple per partition):
  key=partition_result, sum=6, count=3
  key=partition_result, sum=15, count=3
  key=partition_result, sum=24, count=3
  key=partition_result, sum=33, count=3


In [15]:
# mapPartitionsWithIndex — also gives you the partition index
def label_partition(idx, iterator):
    """Labels each element with its partition index."""
    return ((idx, item) for item in iterator)

labeled = data.mapPartitionsWithIndex(label_partition).collect()
print("mapPartitionsWithIndex — each element labeled with its partition:")
for partition_id, value in labeled:
    print(f"  Partition {partition_id}: {value}")

mapPartitionsWithIndex — each element labeled with its partition:
  Partition 0: 1
  Partition 0: 2
  Partition 0: 3
  Partition 1: 4
  Partition 1: 5
  Partition 1: 6
  Partition 2: 7
  Partition 2: 8
  Partition 2: 9
  Partition 3: 10
  Partition 3: 11
  Partition 3: 12


## Part 7: `sample()` and `coalesce()`

In [16]:
big_rdd = sc.parallelize(range(1, 101), 4)

# sample(withReplacement, fraction, seed)
# withReplacement=False → each element selected at most once
# fraction=0.2 → roughly 20% of elements
sample_rdd = big_rdd.sample(withReplacement=False, fraction=0.2, seed=42)
print(f"Original count: {big_rdd.count()}")
print(f"Sample count (~20%): {sample_rdd.count()}")
print(f"Sample: {sorted(sample_rdd.collect())}")

# coalesce: reduce partitions WITHOUT shuffle
print(f"\nBefore coalesce: {big_rdd.getNumPartitions()} partitions")
coalesced = big_rdd.coalesce(2)   # no shuffle — just merges partition data
print(f"After coalesce(2): {coalesced.getNumPartitions()} partitions")
print(f"Data unchanged: {coalesced.count()} elements")

Original count: 100
Sample count (~20%): 30


Sample: [1, 4, 5, 11, 15, 18, 19, 32, 33, 34, 35, 36, 38, 42, 43, 45, 46, 47, 49, 53, 57, 67, 79, 80, 82, 85, 87, 93, 95, 99]

Before coalesce: 4 partitions


After coalesce(2): 2 partitions
Data unchanged: 100 elements


## Part 8: Chaining Transformations — The Real-World Pattern

In [17]:
# Real-world pipeline: process e-commerce order data
raw_orders = sc.parallelize([
    "order_1,alice,electronics,299.99,completed",
    "order_2,bob,clothing,49.99,cancelled",
    "order_3,carol,electronics,599.99,completed",
    "order_4,dave,clothing,89.99,completed",
    "order_5,alice,food,24.99,completed",
    "order_6,bob,electronics,1299.99,completed",
    "order_7,carol,food,15.99,cancelled",
    "order_8,dave,electronics,449.99,completed",
    "order_9,alice,clothing,79.99,completed",
    "order_10,bob,food,34.99,completed",
], 3)

# Pipeline: parse → filter → transform → key-value pairs
completed_revenue = (
    raw_orders
    # Parse CSV
    .map(lambda line: line.split(","))
    # Filter: only completed orders
    .filter(lambda fields: fields[4] == "completed")
    # Extract: (category, revenue)
    .map(lambda fields: (fields[2], float(fields[3])))
    # Sum by category (shuffle!)
    .reduceByKey(lambda a, b: a + b)
    # Sort by revenue descending
    .sortBy(lambda x: x[1], ascending=False)
)

print("Revenue by category (completed orders only):")
for category, revenue in completed_revenue.collect():
    print(f"  {category:<15} ${revenue:,.2f}")

Revenue by category (completed orders only):
  electronics     $2,649.96
  clothing        $169.98
  food            $59.98


## Interview Cheat Sheet

**Q: What is the difference between `map()` and `flatMap()`?**
> `map()` applies a function to each element and returns exactly one output per input, maintaining the structure (a list of lists if the function returns a list). `flatMap()` applies the same function but flattens the result by one level — ideal when your function returns a list and you want a flat collection. Classic use: `flatMap(lambda line: line.split())` for tokenizing text.

**Q: Is `distinct()` a narrow or wide transformation? Why?**
> Wide. To remove duplicates, Spark must compare ALL occurrences of each value across ALL partitions. Since the same value could be in any partition, data must be shuffled so all occurrences of each value end up on the same executor for comparison.

**Q: When would you use `mapPartitions()` instead of `map()`?**
> When the transformation has heavy initialization that should happen once per partition, not once per element. The classic example: writing to a database. With `map()`, you'd open and close a DB connection for every row. With `mapPartitions()`, you open the connection once for the entire partition (thousands of rows), dramatically reducing connection overhead.

**Q: Does `union()` remove duplicates?**
> No. `union()` is purely additive — it combines both RDDs including any common elements. If you need deduplication, chain `.distinct()` after union: `rdd1.union(rdd2).distinct()`.

**Q: What is `coalesce()` and how does it differ from `repartition()`?**
> `coalesce(n)` reduces partitions by merging adjacent ones — it's a narrow transformation with no shuffle, so it's fast. `repartition(n)` does a full shuffle to either increase or decrease partitions, giving a more balanced distribution. Use `coalesce` when reducing partitions before a write; use `repartition` when you need balanced data distribution or want to increase partition count.

## Exercises

1. Take an RDD of `["Hello World", "Spark Is Fast", "Learn RDDs Today"]`. Use `flatMap` to get all words, then `map` to lowercase them, then `filter` to keep words longer than 4 characters.

2. Take `sc.parallelize(range(1, 51), 5)`. Use `map` to create tuples of (number, number²), then `filter` to keep only those where number² > 500.

3. Create two RDDs: one with even numbers 2-20, one with multiples of 3 from 3-30. Use `union` then `distinct` to get unique numbers that are either even OR multiple of 3.

4. Write a `mapPartitions` function that computes the min, max, and count per partition and returns a summary tuple.

In [18]:
# Exercise 1: flatMap → map → filter pipeline
sentences = sc.parallelize(["Hello World", "Spark Is Fast", "Learn RDDs Today"])

result = (
    sentences
    .flatMap(lambda s: s.split())
    .map(lambda w: w.lower())
    .filter(lambda w: len(w) > 4)
)
print("Exercise 1:", result.collect())

Exercise 1: ['hello', 'world', 'spark', 'learn', 'today']


In [19]:
# Exercise 3: union + distinct
evens = sc.parallelize(range(2, 21, 2))           # 2,4,6,...,20
mult3 = sc.parallelize(range(3, 31, 3))           # 3,6,9,...,30

unique_combined = evens.union(mult3).distinct()
print("Exercise 3:", sorted(unique_combined.collect()))

# Exercise 4: mapPartitions summary
data = sc.parallelize(range(1, 25), 4)

def partition_summary(iterator):
    items = list(iterator)
    return iter([(min(items), max(items), len(items))])

results = data.mapPartitions(partition_summary).collect()
print("\nExercise 4 — per-partition summary (min, max, count):")
for i, (mn, mx, cnt) in enumerate(results):
    print(f"  Partition {i}: min={mn}, max={mx}, count={cnt}")

Exercise 3: [2, 3, 4, 6, 8, 9, 10, 12, 14, 15, 16, 18, 20, 21, 24, 27, 30]

Exercise 4 — per-partition summary (min, max, count):
  Partition 0: min=1, max=6, count=6
  Partition 1: min=7, max=12, count=6
  Partition 2: min=13, max=18, count=6
  Partition 3: min=19, max=24, count=6
